# SAM 3 person + shoe tracking danger-zone alarm
사람과 신발을 각각 추적하고, 신발 중심점을 사람 마스크와 매칭한 뒤 danger zone 진입 시 알람을 발생시킵니다.


In [ ]:
import importlib.util, json
from pathlib import Path
import cv2, numpy as np, sam3, torch
from IPython.display import Video, display
from sam3.model_builder import build_sam3_video_predictor
from sam3.visualization_utils import load_frame, render_masklet_frame
root=Path(sam3.__file__).resolve().parent.parent
spec=importlib.util.spec_from_file_location('methods',root/'examples'/'polygon_dangerzone_methods.py'); methods=importlib.util.module_from_spec(spec); spec.loader.exec_module(methods)
video_path=root/'assets'/'videos'/'factory_2.mp4'; zone_path=root/'examples'/'danger_zone_factory_2.json'
out_path=root/'examples'/'sam3_person_shoe_danger_zone.mp4'; summary_path=root/'examples'/'sam3_person_shoe_alarm_summary.json'
frames=methods.load_video_frames(video_path); fps=methods.get_video_fps(video_path)
zone=methods.load_zone_config(zone_path) if zone_path.exists() else methods.select_polygon_on_frame(load_frame(frames[0]))
if not zone_path.exists(): methods.save_zone_config(zone_path,zone,video_path)


In [ ]:
def propagate(p,s):
    out={}
    for r in p.handle_stream_request(request={'type':'propagate_in_video','session_id':s}): out[r['frame_index']]=r['outputs']
    return out
def track(p,text):
    s=p.handle_request(request={'type':'start_session','resource_path':str(video_path)})['session_id']
    p.handle_request(request={'type':'reset_session','session_id':s})
    p.handle_request(request={'type':'add_prompt','session_id':s,'frame_index':0,'text':text})
    out=propagate(p,s); p.handle_request(request={'type':'close_session','session_id':s}); return out
def mask_data(mask):
    m=np.squeeze(mask).astype(bool); y,x=np.where(m)
    if len(x)==0: return None
    return m,(int(np.mean(x)),int(np.mean(y))),(int(x.min()),int(y.min()),int(x.max()),int(y.max()))
def owner(shoe,people):
    d=mask_data(shoe)
    if d is None: return None
    _,(x,y),_=d; cand=[]
    for pid,pmask in people:
        q=mask_data(pmask)
        if q is None: continue
        pm,_,b=q; inside=0<=y<pm.shape[0] and 0<=x<pm.shape[1] and pm[y,x]
        near=b[0]-20<=x<=b[2]+20 and b[1]<=y<=b[3]+20
        if inside or near: cand.append((0 if inside else 1,(x-(b[0]+b[2])/2)**2+(y-b[3])**2,int(pid)))
    return min(cand)[2] if cand else None
predictor=build_sam3_video_predictor(gpus_to_use=range(torch.cuda.device_count()))
people=track(predictor,'person'); shoes=track(predictor,'shoe')


In [ ]:
f0=load_frame(frames[0]); h,w=f0.shape[:2]; poly=methods.normalized_polygon_to_pixels(zone,w,h); contour=poly.astype(np.float32).reshape((-1,1,2))
writer=cv2.VideoWriter(str(out_path),cv2.VideoWriter_fourcc(*'mp4v'),fps,(w,h)); alarms=[]; was_inside={}
try:
  for idx in sorted(shoes):
    so=shoes[idx]; po=people.get(idx,{'out_obj_ids':[],'out_binary_masks':[]}); people_here=list(zip(po['out_obj_ids'],po['out_binary_masks'])); img=render_masklet_frame(load_frame(frames[idx]),so,frame_idx=idx,alpha=.5); active=[]
    for sid,sm in zip(so['out_obj_ids'],so['out_binary_masks']):
      sd=mask_data(sm)
      if sd is None: continue
      _,center,_=sd; pid=owner(sm,people_here)
      if pid is None: continue
      inside=cv2.pointPolygonTest(contour,center,True)>8; key=(int(pid),int(sid)); old=was_inside.get(key,False); was_inside[key]=inside
      color=(255,0,0) if inside else (0,255,0); cv2.circle(img,center,8,color,-1); cv2.putText(img,'P%d-S%d'%(pid,sid),(center[0]+8,center[1]),cv2.FONT_HERSHEY_SIMPLEX,.55,color,2,cv2.LINE_AA)
      if inside: active.append(key)
      if inside and not old:
        event={'frame_idx':idx,'time_seconds':idx/fps,'person_id':int(pid),'shoe_id':int(sid),'shoe_center':center}; alarms.append(event); print('\a[ALARM]',event)
    zc=(255,0,0) if active else (255,215,0); ov=img.copy(); cv2.fillPoly(ov,[poly],zc); img=cv2.addWeighted(ov,.18,img,.82,0); cv2.polylines(img,[poly],True,zc,4)
    if active: cv2.putText(img,'ALARM: shoe in danger zone',(25,55),cv2.FONT_HERSHEY_SIMPLEX,.9,(255,255,255),3,cv2.LINE_AA)
    writer.write(cv2.cvtColor(img,cv2.COLOR_RGB2BGR))
finally: writer.release()
summary_path.write_text(json.dumps(alarms,indent=2,ensure_ascii=False),encoding='utf-8'); print('Saved:',out_path); print('Alarms:',len(alarms)); display(Video(str(out_path),embed=False)); predictor.shutdown()
